# Computer Vision with TensorFlow

Welcome to the **Computer Vision** training notebook! This notebook is designed as a hands-on tutorial to teach you the fundamentals of image processing and neural network design in TensorFlow.

## What is Computer Vision?
Computer Vision (CV) is a field of artificial intelligence that trains computers to interpret and understand the visual world. Using digital images from cameras and videos and deep learning models, machines can accurately identify and classify objects—and then react to what they 'see'.

## Representing Images as Tensors
In deep learning, an image is represented as a **3D Tensor** with dimensions:
$$\text{Shape} = [\text{Height}, \text{Width}, \text{Color Channels}]$$

* **Height:** Number of vertical pixels.
* **Width:** Number of horizontal pixels.
* **Color Channels:** 
  * `1` for **Grayscale** images (representing intensity from 0 to 255).
  * `3` for **RGB (Red, Green, Blue)** color images.

When passed to a neural network, we add a fourth dimension representing the **Batch Size**:
$$\text{Batch Shape} = [\text{Batch Size}, \text{Height}, \text{Width}, \text{Color Channels}]$$

## Dense vs. Convolutional Neural Networks
1. **Dense (Fully Connected) Networks:** Flatten images into 1D vectors before processing. **Drawback:** This discards all spatial relationships (e.g. pixels next to each other are no longer connected).
2. **Convolutional Neural Networks (CNNs):** Slide mathematical filters over the 2D pixel grid. **Advantage:** CNNs retain spatial structure, allowing the model to detect local features like edges, shapes, and complex patterns.


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

print(f"TensorFlow Version: {tf.__version__}")


## 1. Loading & Preprocessing the Data

We will work with the **Fashion MNIST** dataset, containing 70,000 grayscale images of 10 clothing categories. Each image is $28 \times 28$ pixels ($1$ color channel).

### Normalization
Raw pixel values range from `0` to `255`. Dividing them by `255.0` scales them to the range `[0.0, 1.0]`. This feature scaling ensures stable gradients and much faster convergence.


In [ ]:
# Load dataset from Keras built-ins
from tensorflow.keras.datasets import fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# Check raw shape and values
print(f"Train images shape: {train_images.shape}")
print(f"Test images shape: {test_images.shape}")
print(f"Min pixel value: {train_images.min()}, Max pixel value: {train_images.max()}")

# 1. Normalize data to [0.0, 1.0]
train_images = train_images / 255.0
test_images = test_images / 255.0
print(f"After normalization - Min: {train_images.min()}, Max: {train_images.max()}")

# Define class names corresponding to labels 0-9
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


## 2. Visualizing the Data
Let's write a small Matplotlib script to visualize a grid of images along with their labels to verify the dataset loading.


In [ ]:
plt.figure(figsize=(10, 10))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    # Display image
    plt.imshow(train_images[i], cmap=plt.cm.binary)
    # Title with true class
    plt.xlabel(class_names[train_labels[i]])
plt.suptitle("Fashion MNIST Sample Grid", fontsize=16)
plt.show()


## 3. The Anatomy of a CNN Layer
Before building the model, let's understand the core building blocks of a CNN:

### A. Convolutional Layer (`tf.keras.layers.Conv2D`)
The 2D convolutional layer slides small matrices (called **filters** or **kernels**) across the input image. At each step, it performs matrix multiplication and sums the values to produce a single value in the output **feature map**.

**Key Hyperparameters:**
* **`filters`:** Number of kernels to slide across the image. More filters mean the layer can extract more types of features (e.g., vertical edges, horizontal edges, textures).
* **`kernel_size`:** The size of the sliding matrix. Typically $3 \times 3$ or $5 \times 5$.
* **`strides`:** The step size the kernel takes as it slides (default is `1` pixel).
* **`padding`:** 
  * `"valid"`: No zero-padding. The output dimensions shrink: $O = (I - K) + 1$.
  * `"same"`: Zero-padding is applied to the borders so that the output has the *same* height/width as the input.
* **`activation`:** Non-linear function (usually `"relu"`) applied to output features.


### B. Max Pooling Layer (`tf.keras.layers.MaxPool2D`)
Pooling layers are used to downsample (shrink) the spatial dimensions (Height, Width) of the feature maps, reducing computational complexity and overfitting.

* **Mechanism:** A window (usually $2 \times 2$) slides across the feature map, and only the **maximum value** in that window is passed to the next layer.
* **Result:** Halves both Height and Width (reducing total parameters for downstream layers by 75%), while keeping the most dominant features intact.


## 4. Building a Baseline CNN (TinyVGG Style)
Let's build a simple CNN model. Because grayscale images lack the third channel in their raw load shape (loaded as `(28, 28)` instead of `(28, 28, 1)`), we must explicitly reshape or expand dimensions. In Keras, we can pass `(28, 28, 1)` to the `Input` layer.


In [ ]:
tf.random.set_seed(42)

# Define baseline CNN
baseline_model = tf.keras.Sequential([
    # Input layer representing a 28x28 grayscale image
    tf.keras.layers.Input(shape=(28, 28, 1)),
    
    # First Convolutional Block
    tf.keras.layers.Conv2D(filters=10, kernel_size=3, strides=1, padding="valid", activation="relu"),
    tf.keras.layers.MaxPool2D(pool_size=2),
    
    # Second Convolutional Block
    tf.keras.layers.Conv2D(filters=10, kernel_size=3, activation="relu"),
    tf.keras.layers.MaxPool2D(pool_size=2),
    
    # Flatten and Dense Classifier
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation="softmax") # 10 units for 10 classification categories
], name="Baseline_CNN")

baseline_model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                       optimizer=tf.keras.optimizers.Adam(),
                       metrics=["accuracy"])

baseline_model.summary()


## 5. Walkthrough of Model Sizing & Parameter Mathematics

Let's trace how shapes change and parameter counts are calculated based on `baseline_model.summary()`:

### Step 1: Conv2D (1st layer)
* **Input Shape:** `(28, 28, 1)`
* **Formula for Output Shape (valid padding):** $O = I - K + 1 = 28 - 3 + 1 = 26$.
* **Output Shape:** `(26, 26, 10)` (Height and Width are 26, 10 feature maps).
* **Parameters Math:** Each kernel has size $3 \times 3 \times 1$ (width, height, input channels). Plus 1 bias per filter. 
  $$\text{Params} = (\text{Kernel Height} \times \text{Kernel Width} \times \text{Input Channels} + 1) \times \text{Filters}$$
  $$\text{Params} = (3 \times 3 \times 1 + 1) \times 10 = 100 \text{ parameters}.$$

### Step 2: MaxPool2D (2nd layer)
* **Input Shape:** `(26, 26, 10)`
* **Pool Size:** `2` (divides H and W by 2).
* **Output Shape:** `(13, 13, 10)`
* **Parameters:** `0` (MaxPooling has no learnable weights!).

### Step 3: Conv2D (3rd layer)
* **Input Shape:** `(13, 13, 10)`
* **Output Shape (valid padding):** $13 - 3 + 1 = 11$. Shape: `(11, 11, 10)`.
* **Parameters Math:** Each kernel has size $3 \times 3 \times 10$ (since input channel count is now 10).
  $$\text{Params} = (3 \times 3 \times 10 + 1) \times 10 = (90 + 1) \times 10 = 910 \text{ parameters}.$$

### Step 4: MaxPool2D (4th layer)
* **Input Shape:** `(11, 11, 10)`
* **Output Shape:** `(5, 5, 10)` (downsamples 11 to 5 using integer division).

### Step 5: Flatten (5th layer)
* **Input Shape:** `(5, 5, 10)`
* **Output Shape:** $5 \times 5 \times 10 = 250$ values.

### Step 6: Dense Classifier (Output)
* **Input Shape:** `250` features.
* **Output Shape:** `10` output logits.
* **Parameters Math:** Each of the 10 neurons is connected to 250 input features, plus 1 bias per neuron.
  $$\text{Params} = (250 \times 10) + 10 = 2510 \text{ parameters}.$$

### Total Trainable Parameters
$$\text{Total} = 100 + 0 + 910 + 0 + 0 + 2510 = 3520 \text{ parameters}.$$


In [ ]:
# Fit the baseline model
baseline_history = baseline_model.fit(train_images, train_labels,
                                      epochs=5, # Keep it low for fast training demo
                                      validation_data=(test_images, test_labels))


## 6. Plotting Training Curves
Let's write a reusable plotting function to plot the loss and accuracy metrics recorded in the `History` object.


In [ ]:
def plot_curves(history):
    """
    Plots training and validation loss and accuracy curves.
    """
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    accuracy = history.history['accuracy']
    val_accuracy = history.history['val_accuracy']
    
    epochs = range(len(loss))
    
    plt.figure(figsize=(12, 5))
    
    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, loss, label='Training Loss')
    plt.plot(epochs, val_loss, label='Validation Loss')
    plt.title('Loss Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    
    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, accuracy, label='Training Accuracy')
    plt.plot(epochs, val_accuracy, label='Validation Accuracy')
    plt.title('Accuracy Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    
    plt.show()

# Display baseline curves
plot_curves(baseline_history)


## 7. Improving the Model: Stacking Conv2D Layers (VGG Architecture Style)
A standard pattern in Computer Vision is to **double the number of filters** after pooling. By halving spatial resolution (Height, Width), we can afford to increase depth (Channels) without blowing up the compute cost.

Let's build a deeper CNN that stacks multiple `Conv2D` layers before a pooling step. Stacking multiple small kernels ($3 \times 3$) is mathematically superior to using a single large kernel ($5 \times 5$) because it introduces more non-linearities and reduces total parameters.


In [ ]:
tf.random.set_seed(42)

# Build a Deeper VGG-style CNN
vgg_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    
    # Conv Block 1
    tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
    tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPool2D(2),
    
    # Conv Block 2 (Double filters to 32, shrink dimensions)
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPool2D(2),
    
    # Classifier Head
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
], name="VGG_Style_CNN")

vgg_model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  metrics=["accuracy"])

vgg_model.summary()

# Fit model (5 epochs for demo comparison)
vgg_history = vgg_model.fit(train_images, train_labels,
                            epochs=5,
                            validation_data=(test_images, test_labels))


In [ ]:
# Display curves for the deeper VGG-style model
plot_curves(vgg_history)


## 8. Model Evaluation & Visualizing Predictions
Let's write a utility function to plot a random test sample, showing the predicted label (green if correct, red if wrong), its probability confidence, and a side-bar chart of class probabilities.


In [ ]:
import random

def plot_prediction_analysis(model, images, labels, class_names):
    """
    Selects a random test image, makes predictions, and visualizes outcomes.
    """
    idx = random.randint(0, len(images) - 1)
    img = images[idx]
    true_lbl = labels[idx]
    
    # Make prediction (add batch dimension first)
    pred_probs = model.predict(np.expand_dims(img, axis=0))[0]
    pred_lbl = np.argmax(pred_probs)
    
    plt.figure(figsize=(10, 4))
    
    # Plot Image
    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap=plt.cm.binary)
    plt.xticks([])
    plt.yticks([])
    title_color = "green" if pred_lbl == true_lbl else "red"
    plt.title(f"True: {class_names[true_lbl]}\nPred: {class_names[pred_lbl]} ({pred_probs[pred_lbl]*100:.1f}%)",
              color=title_color)
    
    # Plot Probabilities
    plt.subplot(1, 2, 2)
    y_pos = np.arange(len(class_names))
    plt.barh(y_pos, pred_probs, align='center', color='skyblue')
    plt.yticks(y_pos, class_names)
    plt.xlabel('Probability')
    plt.title('Prediction Confidence')
    plt.xlim(0, 1)
    
    plt.tight_layout()
    plt.show()

# Test our visualization utility
plot_prediction_analysis(vgg_model, test_images, test_labels, class_names)


## 9. Summary & Self-Practice Exercises
Congratulations! You've successfully built, analyzed, and evaluated Convolutional Neural Networks for Computer Vision using TensorFlow.

### Takeaways:
1. **Tensors:** Images are stored as 3D arrays; Batching adds a 4th dimension.
2. **Convolution:** Spatial features are extracted using sliding kernels without flattening.
3. **Max Pooling:** Shrinks size and parameters while keeping dominant features.
4. **VGG Rule:** Double filters as you downsample spatial resolution to keep representation capacity high.

### Exercises to Practice:
* **Exercise 1:** Modify the VGG-style CNN to use `padding="valid"` instead of `"same"` in all `Conv2D` layers. Run `model.summary()` and manually calculate the final output shape before the Flatten layer.
* **Exercise 2:** Introduce a **Dropout** layer (`tf.keras.layers.Dropout(0.25)`) right after the first `MaxPool2D` layer. How does it affect validation performance?
* **Exercise 3:** Re-run the VGG model for 10 epochs. Does the training loss continue to drop, and does validation accuracy improve?
